# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 04b: Late Payment Flagging (Schedulable Data Transformation)

---

### What This Notebook Does
Identifies credit card accounts and loans with overdue/upcoming payments and flags them for collections or reminders. This is a **data transformation** notebook designed to run on a recurring schedule.

**Output Table:** `LATE_PAYMENT_FLAGS`

> **Warehouse:** `HOL_MEDIUM_WH`

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_MEDIUM_WH;

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F

session = get_active_session()

## Step 1: Flag Credit Card Late Payments

Identify credit cards where payment is overdue or due within 3 days, and the customer does NOT have autopay enabled.

In [ ]:
# =============================================================================
# CREDIT CARD LATE PAYMENT FLAGS
# =============================================================================

credit_cards_df = session.table("CREDIT_CARDS")
customers_df = session.table("CUSTOMERS")

# Calculate days until payment due
cc_flags = credit_cards_df.filter(
    (F.col("STATUS") == F.lit("ACTIVE")) & 
    (F.col("CURRENT_BALANCE") > F.lit(0))
).with_column(
    "DAYS_UNTIL_DUE",
    F.datediff("day", F.current_date(), F.to_date(F.col("PAYMENT_DUE_DATE")))
).with_columns(
    ["FLAG_TYPE", "URGENCY"],
    [
        F.when(F.col("DAYS_UNTIL_DUE") < F.lit(0), F.lit("OVERDUE"))
         .when((F.col("DAYS_UNTIL_DUE") <= F.lit(3)) & (F.col("AUTOPAY_ENABLED") == False), F.lit("DUE_SOON_NO_AUTOPAY"))
         .when(F.col("DAYS_UNTIL_DUE") <= F.lit(7), F.lit("UPCOMING"))
         .otherwise(F.lit("OK")),
        F.when(F.col("DAYS_UNTIL_DUE") < F.lit(-7), F.lit("CRITICAL"))
         .when(F.col("DAYS_UNTIL_DUE") < F.lit(0), F.lit("HIGH"))
         .when(F.col("DAYS_UNTIL_DUE") <= F.lit(3), F.lit("MEDIUM"))
         .otherwise(F.lit("LOW"))
    ]
).filter(
    F.col("FLAG_TYPE") != F.lit("OK")
).join(
    customers_df.select("CUSTOMER_ID", "FIRST_NAME", "LAST_NAME", "EMAIL", "SEGMENT"),
    "CUSTOMER_ID"
).select(
    F.lit("CREDIT_CARD").alias("PRODUCT_TYPE"),
    F.col("CARD_ID").alias("PRODUCT_ID"),
    F.col("CUSTOMER_ID"),
    F.col("FIRST_NAME"), F.col("LAST_NAME"), F.col("EMAIL"),
    F.col("SEGMENT"),
    F.col("CURRENT_BALANCE").alias("AMOUNT_DUE"),
    F.col("MINIMUM_PAYMENT"),
    F.col("PAYMENT_DUE_DATE"),
    F.col("DAYS_UNTIL_DUE"),
    F.col("FLAG_TYPE"),
    F.col("URGENCY"),
    F.col("AUTOPAY_ENABLED")
)

print(f"Credit card flags: {cc_flags.count()} accounts flagged")
cc_flags.group_by("FLAG_TYPE", "URGENCY").count().sort("URGENCY").show()

## Step 2: Flag Loan Delinquencies

Identify loans in delinquent status that need collections action.

In [ ]:
# =============================================================================
# LOAN DELINQUENCY FLAGS
# =============================================================================

loans_df = session.table("LOANS")

loan_flags = loans_df.filter(
    F.col("STATUS").isin(["DELINQUENT_30", "DELINQUENT_60", "DELINQUENT_90", "DEFAULT"])
).with_columns(
    ["FLAG_TYPE", "URGENCY"],
    [
        F.when(F.col("STATUS") == F.lit("DEFAULT"), F.lit("DEFAULTED"))
         .when(F.col("STATUS") == F.lit("DELINQUENT_90"), F.lit("DELINQUENT_90_DAYS"))
         .when(F.col("STATUS") == F.lit("DELINQUENT_60"), F.lit("DELINQUENT_60_DAYS"))
         .otherwise(F.lit("DELINQUENT_30_DAYS")),
        F.when(F.col("STATUS") == F.lit("DEFAULT"), F.lit("CRITICAL"))
         .when(F.col("STATUS") == F.lit("DELINQUENT_90"), F.lit("CRITICAL"))
         .when(F.col("STATUS") == F.lit("DELINQUENT_60"), F.lit("HIGH"))
         .otherwise(F.lit("MEDIUM"))
    ]
).join(
    customers_df.select("CUSTOMER_ID", "FIRST_NAME", "LAST_NAME", "EMAIL", "SEGMENT"),
    "CUSTOMER_ID"
).select(
    F.lit("LOAN").alias("PRODUCT_TYPE"),
    F.col("LOAN_ID").alias("PRODUCT_ID"),
    F.col("CUSTOMER_ID"),
    F.col("FIRST_NAME"), F.col("LAST_NAME"), F.col("EMAIL"),
    F.col("SEGMENT"),
    F.col("MONTHLY_PAYMENT").alias("AMOUNT_DUE"),
    F.col("MONTHLY_PAYMENT").alias("MINIMUM_PAYMENT"),
    F.col("MATURITY_DATE").alias("PAYMENT_DUE_DATE"),
    F.lit(None).cast("INTEGER").alias("DAYS_UNTIL_DUE"),
    F.col("FLAG_TYPE"),
    F.col("URGENCY"),
    F.lit(False).alias("AUTOPAY_ENABLED")
)

print(f"Loan flags: {loan_flags.count()} loans flagged")
loan_flags.group_by("FLAG_TYPE", "URGENCY").count().sort("URGENCY").show()

## Step 3: Combine and Write Results

In [ ]:
# Union credit card and loan flags, add timestamp
all_flags = cc_flags.union_all(loan_flags).with_column(
    "FLAGGED_AT", F.current_timestamp()
)

all_flags.write.mode("overwrite").save_as_table("LATE_PAYMENT_FLAGS")

result = session.table("LATE_PAYMENT_FLAGS")
print(f"✅ LATE_PAYMENT_FLAGS table: {result.count()} total flags")
result.group_by("PRODUCT_TYPE", "URGENCY").count().sort("PRODUCT_TYPE", "URGENCY").show()